# Gastro decomposition ablation on Kaggle

Trains the decomposition from [mou58/Gastro `utils/`](https://github.com/mou58/Gastro/tree/master/utils): each frame is flattened row by row into one long 1D signal, split into IMFs with 1D EMD (`emd.sift.sift`), and the IMFs are reshaped back to the image. Conditions:

- `original` (baseline)
- `subtract_remd_0`, `subtract_remd_1`, `subtract_remd_0_1`: remove the finest IMFs
- `subtract_remd_trend`: remove IMFs `[-1, -2, -3]`, the three slowest (Gastro's own setting)

The earlier 1D EMD ablation used the same decomposition but scaled each IMF to [0, 1] before subtracting it from the raw MRI, which barely changed the input. Here the IMFs are removed at their real size.

Every condition runs at **seeds 42, 43 and 44**, so the results show run-to-run noise. Same split, epochs, model, loss and three frame exclusions as the multiscale run.

Enable a **GPU** and **Internet**, attach only the raw MRI dataset (CMR-MULTI-CINE-4CH), and run the cells in order. Only the training cell trains.

In [ ]:
from pathlib import Path
import subprocess
import sys

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
assert INPUT.is_dir() and WORK.is_dir(), "Run this notebook on Kaggle."

REPO = WORK / "cardiac-mri-4ch-segmentation"
COMMIT = "SET_AFTER_PUSH"  # commit containing configs/raster_emd_ablation.yaml
URL = "https://github.com/sancia-jw/cardiac-mri-4ch-segmentation.git"
assert COMMIT != "SET_AFTER_PUSH", "Pin COMMIT to the pushed commit first."

if not REPO.exists():
    subprocess.run(["git", "clone", URL, str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", COMMIT], check=True)

head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == COMMIT, f"Existing checkout is {head}; expected {COMMIT}."
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements.txt")], check=True)
print("Repository and dependencies ready:", head)

Locate the raw MRI. Leave the override as None when one matching dataset is attached. DATA must contain 4CH_TR/image and 4CH_TR/anno.

The derived-image cache goes to /tmp, so it is **not** saved as notebook output. It is shared by the three seeds, so the decomposition (about 30 minutes for all four conditions on 4 CPUs) only runs once.

In [ ]:
RAW_OVERRIDE = None  # Example: "/kaggle/input/cmr-multi-cine-4ch"

matches = sorted({p.parent for p in INPUT.rglob("4CH_TR")
                  if (p / "image").is_dir() and (p / "anno").is_dir()})
if RAW_OVERRIDE:
    DATA = Path(RAW_OVERRIDE)
elif len(matches) == 1:
    DATA = matches[0]
else:
    raise RuntimeError(f"Expected one raw MRI root; found {[str(p) for p in matches]}. Set RAW_OVERRIDE.")

OUT = WORK / "raster_emd_ablation"
LOGS = WORK / "raster_emd_logs"
ENHANCED = Path("/tmp/raster_emd_enhanced_cache")
CONFIG = REPO / "configs/raster_emd_ablation.yaml"
EXCLUSIONS = REPO / "configs/bemd_exclusions.json"
SPLITS = REPO / "outputs/splits_4ch.csv"

assert (DATA / "4CH_TR/image").is_dir() and (DATA / "4CH_TR/anno").is_dir()
for path in (CONFIG, EXCLUSIONS, SPLITS):
    assert path.is_file(), f"Missing {path}"
for name, path in [("DATA", DATA), ("OUT", OUT), ("ENHANCED (not saved)", ENHANCED)]:
    print(f"{name}: {path}")

Check the GPU, conditions, controls and exclusions. This does not build or train a model.

`SEEDS` defaults to 42, 43, 44: 15 training runs, about 5 hours. If a session can't fit them (Kaggle stops a session at 12 h), set `SEEDS = [42]` here, save a version, then run another version with `SEEDS = [43, 44]`. Each seed writes to its own folder.

In [ ]:
import json
import os
import torch
import yaml

assert torch.cuda.is_available(), "Enable a GPU in Kaggle notebook settings."
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__, "| CPUs:", os.cpu_count())

cfg = yaml.safe_load(CONFIG.read_text())
RUNS = cfg["train"]["runs"]
assert cfg["train"]["catalog"] == "raster_emd"
assert (cfg["train"]["epochs"], cfg["train"]["batch_size"], cfg["train"]["lr"]) == (15, 4, 0.001)
assert RUNS == ["original", "subtract_remd_0", "subtract_remd_1", "subtract_remd_0_1", "subtract_remd_trend"]

exclusions = json.loads(EXCLUSIONS.read_text())
assert {(r["case_stem"], r["frame_idx"]) for r in exclusions["excluded_frames"]} == {
    ("CINE_4CH_023", 16), ("CINE_4CH_051", 65), ("CINE_4CH_085", 65),
}

SEEDS = [42, 43, 44]
os.environ["ONTHEFLY_WORKERS"] = str(os.cpu_count() or 1)  # parallel decomposition; inherited by the CLI
print("Conditions:", RUNS)
print("Seeds:", SEEDS)

Validate inputs with the actual CLI: case split, frame counts and exclusions from the raw headers. No training, evaluation or decomposition.

In [ ]:
LOGS.mkdir(parents=True, exist_ok=True)

def base_command(seed):
    return [
        sys.executable, "-u", str(REPO / "scripts/run_bemd_ablation.py"),
        "--config", str(CONFIG),
        "--data-root", str(DATA),
        "--exclusions", str(EXCLUSIONS),
        "--splits-csv", str(SPLITS),
        "--output-root", str(OUT / f"seed_{seed}"),
        "--enhanced-cache-root", str(ENHANCED),
        "--device", "cuda",
        "--seed", str(seed),
        "--runs", *RUNS,
    ]

def run_logged(command, log_name):
    log_path = LOGS / log_name
    print("Running:", " ".join(command))
    with log_path.open("w", encoding="utf-8") as log:
        with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
    if code:
        raise RuntimeError(f"Command exited with code {code}; see {log_path}")

VALIDATION_PASSED = False
run_logged(base_command(SEEDS[0]) + ["--validate-only"], "validation.log")
VALIDATION_PASSED = True
print("Validation passed. No model has been trained.")

**START TRAINING**: runs every condition at every seed in `SEEDS`, with test evaluation. Refuses to overwrite existing run folders.

In [ ]:
import time

assert VALIDATION_PASSED, "Run the validation cell successfully first."
existing = [OUT / f"seed_{s}" / r for s in SEEDS for r in RUNS if (OUT / f"seed_{s}" / r).exists()]
if existing:
    raise RuntimeError("Refusing to overwrite: " + ", ".join(str(p) for p in existing))

t0 = time.time()
for seed in SEEDS:
    run_logged(base_command(seed), f"training_seed_{seed}.log")
    print(f"Seed {seed} done; {(time.time() - t0) / 3600:.2f} h so far.")
print(f"Completed {len(RUNS)} conditions x {len(SEEDS)} seeds in {(time.time() - t0) / 3600:.2f} h.")

Results. `delta_*` is the condition minus `original` **at the same seed**, then averaged over seeds; `sd` is the spread across seeds. A difference smaller than the spread of `original` itself is noise.

In [ ]:
import pandas as pd
from IPython.display import display, FileLink

FG, MYO = "test_dice_mean_foreground", "test_dice_class_2_Left Ventricle Myocardium"
frames = []
for path in sorted(OUT.glob("seed_*/ablation_summary.csv")):
    frames.append(pd.read_csv(path))
allruns = pd.concat(frames, ignore_index=True)
for metric in (FG, MYO):
    base = allruns[allruns.run_id == "original"].set_index("seed")[metric]
    allruns["delta_" + metric] = allruns[metric] - allruns["seed"].map(base)
allruns.to_csv(OUT / "all_seeds_summary.csv", index=False)

table = allruns.groupby("run_id").agg(
    seeds=("seed", "count"),
    test_fg_mean=(FG, "mean"), test_fg_sd=(FG, "std"),
    delta_fg_mean=("delta_" + FG, "mean"), delta_fg_sd=("delta_" + FG, "std"),
    lv_myo_mean=(MYO, "mean"), delta_myo_mean=("delta_" + MYO, "mean"),
).reindex(RUNS).round(4)
table.to_csv(OUT / "seed_table.csv")
display(table)

display(FileLink(str(OUT / "seed_table.csv")))
display(FileLink(str(OUT / "all_seeds_summary.csv")))
for seed in sorted(allruns["seed"].unique()):
    for run_id in RUNS:
        checkpoint = OUT / f"seed_{seed}" / run_id / "checkpoints/best.pt"
        assert checkpoint.is_file(), f"Missing best checkpoint: {checkpoint}"
print("All best checkpoints present.")